# TSDAE 教學：不用任何標籤，訓練句向量模型

---

## TSDAE 是什麼

**TSDAE** = **T**ransformer-based **S**equential **D**enoising **A**uto**E**ncoder。

它是**非監督**方法：**不需要標籤，也不需要句子配對**，只要一堆句子就能訓練。

### 核心想法：把句子弄壞，再要求模型修好

```
原句：   A man is riding a horse in the park
            ↓ 破壞（隨機刪掉一些詞）
壞句：   man riding horse park
            ↓
        [encoder = BERT]
            ↓
        一個句向量（768 維）          ← 唯一的資訊通道
            ↓
        [decoder]
            ↓
重建：   A man is riding a horse in the park   ← 要和原句越像越好
```

關鍵在中間那個**句向量**：decoder 看不到壞句本身，只能靠這一個向量還原整句。為了讓 decoder 還原得好，encoder 被迫把整句的語意都壓進這個向量裡。訓練完後，我們只留下 encoder，它就成了一個會產生句向量的模型。

### 跟之前學過的方法比較

| 方法 | 需要標籤 / 配對嗎 | 學習訊號 |
|---|---|---|
| SoftmaxLoss | 要（句子對 + 類別標籤） | 分類 |
| CosineSimilarityLoss | 要（句子對 + 相似度） | 回歸 |
| MultipleNegativesRankingLoss | 要（正例配對） | 對比學習 |
| Augmented SBERT | 少量要，其餘用老師模型補 | 回歸 |
| **TSDAE（這份）** | **不需要** | 重建句子 |

TSDAE 的價值：當你的領域**完全沒有標註資料**時，仍然能適應該領域的語言。

## 步驟 1：載入套件與資料

| 套件 / 類別 | 用途 |
|---|---|
| `nltk` | 自然語言處理工具包，這裡用它的分詞器把句子切成詞 |
| `DenoisingAutoEncoderDataset` | 負責「把句子弄壞」，並配成（壞句, 原句）一對 |
| `EmbeddingSimilarityEvaluator` | 用句子相似度資料集評估模型 |
| `tqdm` | 顯示進度條 |

### `nltk.download('punkt_tab')`

`DenoisingAutoEncoderDataset` 要刪詞，就得先知道句子怎麼切成詞，這需要 nltk 的分詞資料。`punkt_tab` 就是那份資料，**只需要下載一次**，之後會留在本機。


### 資料

```python
.select(range(25000))
```

取 MNLI 訓練集前 25000 筆。**這裡不使用標籤**，只拿裡面的句子。MNLI 在這裡只是「一個方便取得英文句子的來源」。

In [ ]:
import nltk 
from tqdm import tqdm
from datasets import Dataset, load_dataset
from sentence_transformers.sentence_transformer.datasets import DenoisingAutoEncoderDataset
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator


nltk.download('punkt_tab')

mnli = load_dataset(
"nyu-mll/glue",
"mnli",
split = "train",
).select(range(25000))

## 步驟 2：把兩欄句子攤平成一個清單（flat）

MNLI 本來是「兩欄」的表格，每列是一對句子：

```
premise              hypothesis
"A man rides a horse"  "Someone rides an animal"
"A girl plays piano"   "Someone is performing"
```

TSDAE 只需要「一堆句子」，不在乎配對關係。所以把兩欄疊成一個長清單：

```python
flat_sentence = list(mnli["premise"]) + list(mnli["hypothesis"])
```

- `list(...)`：新版 `datasets` 取出的欄位是 `Column` 物件，先轉成 list 才能用 `+` 串接
- `+`：把兩個 list 接在一起

結果是約 50000 句（25000 × 2）。`print(flat_sentence[0])` 印出第一句，確認內容是正常的英文句子。

In [ ]:
flat_sentence = list(mnli["premise"]) + list(mnli["hypothesis"])
print(flat_sentence[0])

## 步驟 3：製造「（壞句, 原句）」訓練資料

```python
damaged_data = DenoisingAutoEncoderDataset(list(set(flat_sentence)))
```

### `set(...)` 去重複

MNLI 裡同一個 premise 會配好幾個 hypothesis，所以 `flat_sentence` 有**很多重複的句子**。`set` 去掉重複，避免同一句話被練很多次。

> `set` 不保證順序，所以每次執行的句子順序可能不同。對訓練沒影響（Trainer 本來就會打亂），但如果你需要「完全可重現」，要另外處理。

### `DenoisingAutoEncoderDataset` 做了什麼

它是一個資料集物件，你逐筆取出時，會得到一個 `InputExample`：

```python
data.texts[0]   # 壞句（被刪掉一些詞）
data.texts[1]   # 原句（完整的）
```

每次取出時，它會對原句**隨機刪詞**（我記得預設刪掉相當大的比例，確切預設值我沒有針對你的版本查證）。

### 為什麼要轉成 `Dataset`

```python
for data in tqdm(damaged_data):
    train_dataset["damaged_data"].append(data.texts[0])
    train_dataset["original_data"].append(data.texts[1])

train_dataset = Dataset.from_dict(train_dataset)
```

新版的 `SentenceTransformerTrainer` 吃的是 Hugging Face `Dataset`，不是舊版的 `InputExample`。所以這裡把每一筆拆成兩欄，再用 `from_dict` 轉換：

| 欄位 | 內容 |
|---|---|
| `damaged_data` | 壞句（模型的輸入） |
| `original_data` | 原句（重建的目標） |

**欄位順序很重要**：Trainer 依照欄位的**順序**把資料交給 loss，第一欄當輸入、第二欄當目標，欄位名稱本身倒是無所謂。


In [ ]:
damaged_data = DenoisingAutoEncoderDataset(
list(set(flat_sentence))
)

train_dataset = {
"damaged_data" : [],
"original_data" : [],
}

for data in tqdm(damaged_data):
    train_dataset["damaged_data"].append(data.texts[0])
    train_dataset["original_data"].append(data.texts[1])

train_dataset = Dataset.from_dict(
train_dataset
)

## 步驟 4：檢查資料，並建立 STS-B 評估器

`print(train_dataset[0])` 印出第一筆，你會看到壞句和原句對照，這是**最重要的檢查**：確認壞句真的有被刪詞、原句是完整的。

### 評估器

TSDAE 訓練時**完全沒用到任何標籤**，那怎麼知道訓練得好不好？用 STS-B：

- STS-B 每對句子有人工標註的相似度（0~5）
- `score / 5` 正規化成 0~1
- `main_similarity="cosine"`：算兩句向量的餘弦相似度，再和人工分數算相關係數（越高越好）

訓練用的是「重建句子」，評估用的是「判斷相似度」，是**完全不同的任務**，正好檢驗向量是否有通用的語意。

In [ ]:
print(train_dataset[0])

val_sts = load_dataset(
"nyu-mll/glue",
"stsb",
split = "validation",
)
evaluator = EmbeddingSimilarityEvaluator(
sentences1 = val_sts["sentence1"],
sentences2 = val_sts["sentence2"],
scores = [score / 5 for score in val_sts["label"]],
main_similarity = "cosine"
)

## 步驟 5：手動建立 Bi-Encoder 模型（CLS pooling）

### 為什麼不直接 `SentenceTransformer("bert-base-uncased")`

直接這樣寫，函式庫會自動補一個 **mean pooling** 層。但 TSDAE 要用 **CLS pooling**，所以得自己把兩層組起來：

```
文字 → [Transformer: bert-base-uncased] → [Pooling: cls] → 句向量
```

### 什麼是 pooling

BERT 對每個 token 都輸出一個 768 維向量，句向量只要一個，所以要「壓成一個」：

| pooling_mode | 做法 |
|---|---|
| `"mean"`（預設） | 所有 token 向量取平均 |
| `"cls"` | 只取第一個 `[CLS]` token 的向量 |
| `"max"` | 每個維度取最大值 |

### 為什麼 TSDAE 用 CLS

TSDAE 的 decoder 只靠**一個向量**重建整句，CLS 是專門「代表整句」的位置，用它當瓶頸目標最明確。官方範例也採用 CLS（論文是否有比較過其他 pooling，我沒有重新查證）。

### `embedding_model.max_seq_length = 128`

句子最多只取 128 個 token，超過的截掉。這行是為了**避免 GPU 記憶體不足**：decoder 要對每個位置、每個詞彙（約 30522 個）計算分數，記憶體用量和句長成正比，限制句長可以大幅降低用量。

In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.sentence_transformer.modules import Transformer, Pooling
word_embedding_model = Transformer("bert-base-uncased")
pooling_model = Pooling(
embedding_dimension = word_embedding_model.get_embedding_dimension(),
pooling_mode = "cls",
)
embedding_model = SentenceTransformer(modules = [word_embedding_model, pooling_model])
embedding_model.max_seq_length = 128

## 步驟 6：建立 Loss（含 decoder）

```python
train_loss = losses.DenoisingAutoEncoderLoss(
    model=embedding_model,
    tie_encoder_decoder=True,
)
```

### decoder 是什麼

這個 loss 內部會建立一個 **decoder**，負責用句向量重建原句。它是訓練時的輔助工具：

- **只在訓練時使用**，訓練完不需要
- **不會存進最終模型**，所以最終模型和一般的 BERT + pooling 一樣大

### `tie_encoder_decoder=True`

讓 decoder **共用 encoder 的權重**，不另外保存一份。好處是省參數、省記憶體，也讓 encoder 承擔更多學習責任。

### `train_loss.decoder = train_loss.decoder.to("cuda")`

Trainer 不一定會幫 loss 裡面的 decoder 搬到 GPU，所以手動把它搬過去。不搬的話，encoder 輸出在 GPU、decoder 在 CPU，會出現「tensor 不在同一個 device」的錯誤。

> 這行把裝置寫死為 `"cuda"`，所以**沒有 NVIDIA GPU 的機器會直接報錯**。

In [ ]:
from sentence_transformers.sentence_transformer import losses

train_loss = losses.DenoisingAutoEncoderLoss(
model = embedding_model,
tie_encoder_decoder = True,
)
train_loss.decoder = train_loss.decoder.to("cuda")

## 步驟 7：訓練參數、Trainer、開始訓練

| 參數 | 值 | 說明 |
|---|---|---|
| `output_dir` | `TSADE_embedding_model` | 輸出資料夾（checkpoint、記錄），只是資料夾名稱 |
| `num_train_epochs` | 1 | 整份資料看一遍 |
| `per_device_train_batch_size` | 16 | 每步 16 句。DAE 要 decoder，比一般訓練更吃顯存，所以比前面練習的 32 小 |
| `warmup_steps` | 100 | 學習率前 100 步從 0 升到預設的 `5e-5` |
| `fp16` | True | 混合精度，省顯存、加速（需要支援的 GPU） |
| `eval_steps` / `logging_steps` | 100 | 每 100 步評估 / 記錄 |

### 訓練前後對比

```python
print("before:", evaluator(embedding_model))   # 訓練前
trainer.train()
print("after:", evaluator(embedding_model))    # 訓練後（下一個儲存格）
```

`trainer.train()` 會**原地修改** `embedding_model`，所以前後兩次分數的差，就是 TSDAE 帶來的進步。

### 大概會訓練多少步

去重複後句子數量會比 50000 少（MNLI 的 premise 重複很多），實際筆數用 `len(train_dataset)` 確認。步數約等於「筆數 / 16」。

### 如果遇到 CUDA out of memory

| 做法 | 說明 |
|---|---|
| 降低 `max_seq_length` | 最有效，decoder 輸出和句長成正比 |
| 降低 `per_device_train_batch_size` 並加 `gradient_accumulation_steps` | 例如 4 × 4 = 等效 16 |
| 重啟 Kernel | 互動環境會殘留上一輪的模型和顯存 |
| 在 terminal 用 `python` 執行 | 程式結束，顯存自動釋放 |

In [ ]:
from sentence_transformers.sentence_transformer.training_args import SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer.trainer import SentenceTransformerTrainer

args = SentenceTransformerTrainingArguments(
output_dir = "TSADE_embedding_model",
num_train_epochs = 1,
per_device_train_batch_size = 16,
per_device_eval_batch_size = 16,
warmup_steps = 100,
fp16 = True,
eval_steps = 100,
logging_steps = 100,
)

trainer = SentenceTransformerTrainer(
model = embedding_model,
args = args,
train_dataset = train_dataset,
loss = train_loss,
evaluator = evaluator,
)
print("before:", evaluator(embedding_model))
trainer.train()

## 步驟 8：訓練後評估

和上一步的 `before` 比較，就能看出 TSDAE 有沒有幫助。

預期情況（我沒有實測，只是依方法特性推測）：未訓練的 BERT 當句向量分數偏低，TSDAE 訓練後應該會提升。但要注意 TSDAE 是**非監督**方法，在通用的 STS-B 上，通常不如有標籤的監督式方法（像前面的 NLI 訓練）。它真正的優勢是在**沒有標註資料的特定領域**做適應。

In [ ]:
print("after:", evaluator(embedding_model))

---

# 重點整理

## 一、整個方法一句話

> 把句子隨機刪掉一些詞，讓 encoder 把壞句壓成**一個句向量**，再讓 decoder 靠這個向量還原原句；為了還原得好，encoder 被迫學會把整句語意塞進向量裡。**全程不需要標籤。**

## 二、必須搞懂的 7 個概念

| # | 概念 | 一句話解釋 |
|---|---|---|
| 1 | 非監督 | 只要句子，不要標籤、不要配對 |
| 2 | 破壞 + 重建 | 刪詞後還原，是這個方法的學習訊號 |
| 3 | 句向量瓶頸 | decoder 只能靠一個向量重建，逼 encoder 壓縮語意 |
| 4 | flat（攤平） | 把 premise 和 hypothesis 疊成一個句子清單 |
| 5 | CLS pooling | 取 `[CLS]` 位置的向量當句向量，TSDAE 的標準做法 |
| 6 | decoder | 訓練專用的輔助模組，不存進最終模型 |
| 7 | `tie_encoder_decoder` | decoder 共用 encoder 權重，省參數與記憶體 |

## 三、程式流程速記

```
1. 載入 MNLI 前 25000 筆（只取句子，不用標籤）
2. premise + hypothesis 攤平成一個清單
3. set 去重複 → DenoisingAutoEncoderDataset 製造（壞句, 原句）→ 轉成 HF Dataset
4. 建立 STS-B 評估器
5. 手動組 Transformer + Pooling(cls)，限制 max_seq_length = 128
6. 建立 DenoisingAutoEncoderLoss（tie 權重），decoder 搬到 GPU
7. 設定訓練參數 → before 評估 → trainer.train()
8. after 評估，和 before 比較
```

## 四、最容易搞混的地方

1. **為什麼要 flat？**
   TSDAE 只需要「一堆句子」，不需要配對，所以把兩欄疊成一欄，資料量也翻倍。

2. **為什麼手動建模型，不直接用 `SentenceTransformer("bert-base-uncased")`？**
   直接用會自動套 mean pooling；TSDAE 要 CLS pooling，所以要自己組。

3. **decoder 為什麼不能留在 CPU？**
   encoder 輸出在 GPU，decoder 在 CPU 會因 device 不一致而報錯，就算能跑也極慢。

4. **評估為什麼用 STS-B？**
   訓練用「重建句子」，沒有任何相似度資訊，用 STS-B 才能檢驗向量的語意品質。

5. **訓練需要標籤嗎？**
   不需要。MNLI 的 `label` 在這份程式中從頭到尾都沒用到。

## 五、記憶體相關設定一覽

| 設定 | 作用 |
|---|---|
| `max_seq_length = 128` | 限制句長，降低 decoder 輸出的記憶體 |
| `per_device_train_batch_size = 16` | 每步句子數，越小越省 |
| `fp16 = True` | 混合精度，省記憶體 |
| `tie_encoder_decoder = True` | 共用權重，不用多存一份 decoder |

## 六、觀察到的幾個地方（程式碼未修改，僅供參考）

這些不一定是錯誤，是進階時可以思考的點：

1. **`eval_steps=100` 不會生效**
   只設 `eval_steps` 而沒設 `eval_strategy="steps"`，訓練中預設不會評估。目前只有訓練前後各評估一次。

2. **沒有儲存最終模型**
   `output_dir` 只放 checkpoint，不保證根目錄能直接載入。訓練完建議另外呼叫 `embedding_model.save("某個資料夾")`。

3. **`.to("cuda")` 寫死裝置**
   沒有 NVIDIA GPU 的環境會直接報錯。

4. **破壞只做一次**
   迴圈只跑一輪，每句話的壞版本固定下來，多 epoch 時每輪看到的相同。

5. **`set` 順序不固定**
   每次執行的資料順序可能不同，結果不完全可重現。

6. **GPU 小的限制**
   7.5 GB 的 GPU 跑 BERT-base 加 decoder 很吃緊，要靠限制句長與 batch size 才跑得動。

## 七、可以自己做的實驗（需要時再改，不影響原程式）

```python
# 1. 存下最終模型，之後不用重新訓練
embedding_model.save("tsdae_final_model")

# 2. 比較 pooling：把 pooling_mode 改成 "mean"，看分數差異

# 3. 比較 tie_encoder_decoder=False，看記憶體與分數的變化

# 4. 先用 TSDAE 做無監督預訓練，再接前面學過的 NLI 監督訓練，
#    看兩階段是否比單獨使用任一種更好
```

## 八、一句話記住

> **TSDAE = 弄壞句子 → 壓成一個向量 → 還原句子。** 它讓你在完全沒有標註的情況下，也能訓練出會產生句向量的模型。